<a href="https://colab.research.google.com/github/AsimobiChisomSandra/ATF_Training_Practice/blob/main/Week3_10_tool_use_ai_agents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 3.1: Tool Use - Building AI Agents

## Key Insight: Thought → Action → Observation → Answer

Your digital twin learns to USE TOOLS autonomously!

## What You'll Build
- Tools your AI can call (calculator, search, etc.)
- Agent that decides when to use tools
- Multi-step reasoning loops
- Digital twin with tool capabilities

In [1]:
import json
from typing import Dict, List, Any, Callable
from datetime import datetime

print('Week 3.1: Tool Use & Agents')
print('Goal: Make your digital twin DO things, not just talk!')

Week 3.1: Tool Use & Agents
Goal: Make your digital twin DO things, not just talk!


## Part 1: Creating Tools

A tool is a function your AI can call.

In [2]:
# Tool Example 1: Calculator
def calculator(expression: str) -> Dict[str, Any]:
    '''Calculate math expressions.'''
    try:
        result = eval(expression)  # Safe in controlled environment
        return {'result': result, 'expression': expression}
    except Exception as e:
        return {'error': str(e)}

# Tool Example 2: Get current time
def get_time() -> Dict[str, Any]:
    '''Get current time.'''
    now = datetime.now()
    return {
        'time': now.strftime('%H:%M:%S'),
        'date': now.strftime('%Y-%m-%d'),
        'day': now.strftime('%A')
    }

print('Calculator:', calculator('15 * 0.15'))
print('Time:', get_time())

Calculator: {'result': 2.25, 'expression': '15 * 0.15'}
Time: {'time': '11:22:57', 'date': '2026-10-14', 'day': 'Wednesday'}


## Part 2: Tool Registry

Manage available tools.

In [3]:
class ToolRegistry:
    '''Manages available tools for the agent.'''

    def __init__(self):
        self.tools = {}

    def register(self, name: str, func: Callable, description: str):
        '''Register a tool.'''
        self.tools[name] = {
            'function': func,
            'description': description,
            'name': name
        }

    def get_tool(self, name: str) -> Callable:
        '''Get tool function by name.'''
        return self.tools[name]['function']

    def list_tools(self) -> str:
        '''Get formatted list of available tools.'''
        lines = ['Available tools:']
        for name, info in self.tools.items():
            lines.append(f"- {name}: {info['description']}")
        return '\n'.join(lines)

#Create registry
registry = ToolRegistry()
registry.register('calculator', calculator, 'Calculate math expressions')
registry.register('get_time', get_time, 'Get current date and time')

print(registry.list_tools())

Available tools:
- calculator: Calculate math expressions
- get_time: Get current date and time


## Part 3: ReAct Agent

Agent that can reason and act with tools.

**ReAct Loop:**
1. **Thought**: Decide what to do
2. **Action**: Use a tool
3. **Observation**: See result
4. **Answer**: Respond to user (or loop again)

In [4]:
class SimpleReActAgent:
    '''Simple agent that can use tools.'''

    def __init__(self, tool_registry: ToolRegistry):
        self.registry = tool_registry
        self.history = []

    def think(self, query: str) -> str:
        '''Decide what action to take.'''
        # Simplified: pattern matching for demo
        query_lower = query.lower()

        if any(word in query_lower for word in ['calculate', 'math', '+', '-', '*', '/']):
            return 'use_calculator'
        elif any(word in query_lower for word in ['time', 'date', 'day']):
            return 'use_time'
        else:
            return 'direct_answer'

    def act(self, action: str, query: str) -> Dict[str, Any]:
        '''Execute the action.'''
        if action == 'use_calculator':
            # Extract math expression from query
            import re
            match = re.search(r'[\d+\-*/().\s]+', query)
            if match:
                expr = match.group(0).strip()
                return self.registry.get_tool('calculator')(expr)
            return {'error': 'Could not extract expression'}

        elif action == 'use_time':
            return self.registry.get_tool('get_time')()

        else:
            return {'direct': query}

    def run(self, query: str) -> str:
        '''Run the agent on a query.'''
        # Step 1: Thought
        thought = self.think(query)
        self.history.append({'step': 'thought', 'content': thought})

        # Step 2: Action
        observation = self.act(thought, query)
        self.history.append({'step': 'action', 'content': observation})

        # Step 3: Answer
        if 'result' in observation:
            answer = f"The result is {observation['result']}"
        elif 'time' in observation:
            answer = f"It's {observation['time']} on {observation['date']} ({observation['day']})"
        elif 'error' in observation:
            answer = f"Error: {observation['error']}"
        else:
            answer = "I can help with calculations and time queries."

        return answer

# Test the agent
agent = SimpleReActAgent(registry)

print('\nAgent Tests:')
print('Q: What is 25 * 4?')
print('A:', agent.run('What is 25 * 4?'))
print('\nQ: What time is it?')
print('A:', agent.run('What time is it?'))


Agent Tests:
Q: What is 25 * 4?
A: Error: invalid syntax (<string>, line 0)

Q: What time is it?
A: It's 11:35:24 on 2026-10-14 (Wednesday)


## Part 4: Integrating with Digital Twin

Add tools to your Week 1 digital twin.

In [5]:
class DigitalTwinWithTools:
    '''Digital twin that can use tools.'''

    def __init__(self, name: str):
        self.name = name
        self.registry = ToolRegistry()
        self.agent = None
        self.setup_tools()

    def setup_tools(self):
        '''Setup default tools.'''
        self.registry.register('calculator', calculator, 'Calculate math')
        self.registry.register('get_time', get_time, 'Get current time')
        self.agent = SimpleReActAgent(self.registry)

    def add_tool(self, name: str, func: Callable, description: str):
        '''Add a new tool.'''
        self.registry.register(name, func, description)

    def chat(self, message: str) -> str:
        '''Chat with tool use.'''
        return self.agent.run(message)

# Create digital twin with tools
twin = DigitalTwinWithTools('My Assistant')

print('\nDigital Twin with Tools:')
print(twin.chat('Calculate 47.50 * 0.15'))
print(twin.chat('What day is today?'))


Digital Twin with Tools:
The result is 7.125
It's 11:40:39 on 2026-10-14 (Wednesday)


## Exercise 1: Create Custom Tools

**Task**: Add 2 custom tools to your digital twin.

Ideas:
- Memory search
- Note taker
- Task list
- Habit tracker

In [6]:
import re

# 1. Define our in-memory storage for notes
_notes_db = []

# 2. Tool 1: Note Taker
def take_note(note: str) -> Dict[str, Any]:
    '''Saves a written note to the digital twin's memory database.'''
    if not note.strip():
        return {'error': 'Note content cannot be empty.'}
    _notes_db.append(note.strip())
    return {'result': f"Successfully saved note: '{note.strip()}'"}

# 3. Tool 2: Memory Search
def search_notes(query: str) -> Dict[str, Any]:
    '''Searches saved notes for occurrences of a specific keyword.'''
    keyword = query.strip().lower()
    if not keyword:
        return {'error': 'Search query cannot be empty.'}
    results = [note for note in _notes_db if keyword in note.lower()]
    return {
        'query': query,
        'results': results,
        'count': len(results)
    }

# 4. Extend the SimpleReActAgent to support our new tools
class ExtendedReActAgent(SimpleReActAgent):
    def think(self, query: str) -> str:
        query_lower = query.lower()
        # Prioritize our custom tools
        if any(word in query_lower for word in ['note', 'write down', 'remember that', 'save']):
            return 'use_note_taker'
        elif any(word in query_lower for word in ['search', 'find', 'lookup']):
            return 'use_memory_search'
        # Fallback to standard tools
        return super().think(query)

    def act(self, action: str, query: str) -> Dict[str, Any]:
        if action == 'use_note_taker':
            # Simple extraction: remove command verbs
            note_content = re.sub(r'(?i)(write down|remember that|note|save)\s*', '', query).strip()
            return self.registry.get_tool('take_note')(note_content)

        elif action == 'use_memory_search':
            # Extract the search keyword
            keyword = re.sub(r'(?i)(search for|find|lookup|search)\s*', '', query).strip()
            return self.registry.get_tool('search_notes')(keyword)

        return super().act(action, query)

    def run(self, query: str) -> str:
        # Override run to handle our new observation keys
        thought = self.think(query)
        observation = self.act(thought, query)

        if 'result' in observation:
            return observation['result']
        elif 'results' in observation:
            if observation['count'] == 0:
                return f"I couldn't find any notes matching '{observation['query']}'."
            results_str = '\n'.join([f"- {r}" for r in observation['results']])
            return f"Found {observation['count']} matching note(s):\n{results_str}"

        # Fallback to original agent's formatter
        if 'time' in observation:
            return f"It's {observation['time']} on {observation['date']} ({observation['day']})"
        elif 'error' in observation:
            return f"Error: {observation['error']}"
        else:
            return "I can help you store notes, search them, calculate math, or tell time!"

# 5. Initialize the Twin with the Extended Agent and Register tools
# We will register them directly on the twin's registry
twin.registry.register('take_note', take_note, 'Save a new written note to memory')
twin.registry.register('search_notes', search_notes, 'Search past notes for keywords')

# Override the twin's agent with our Extended Agent
twin.agent = ExtendedReActAgent(twin.registry)

# --- Test the Integration ---
print("=== Demonstration of Custom Tools ===\n")
print("1. Saving a note:")
print("Response:", twin.chat("write down buy milk and bread at the grocery store"))
print("Response:", twin.chat("remember that my meeting with Sarah is at 3 PM"))

print("\n2. Searching for a note containing 'meeting':")
print("Response:", twin.chat("search meeting"))

print("\n3. Searching for a note containing 'milk':")
print("Response:", twin.chat("find milk"))

=== Demonstration of Custom Tools ===

1. Saving a note:
Response: Successfully saved note: 'buy milk and bread at the grocery store'
Response: Successfully saved note: 'my meeting with Sarah is at 3 PM'

2. Searching for a note containing 'meeting':
Response: Found 1 matching note(s):
- my meeting with Sarah is at 3 PM

3. Searching for a note containing 'milk':
Response: Found 1 matching note(s):
- buy milk and bread at the grocery store


## Summary

### What We Learned:
- Tools are functions AI can call
- Tool registry manages available tools
- ReAct pattern: Think → Act → Observe
- Agents can use tools autonomously
- Digital twin + tools = powerful assistant

### Next: Week 3.2 - Testing Your AI